In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_Exploration") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready")

Spark 3.5.1 ready


In [15]:
from pyspark.sql.functions import col
from pyspark.sql.types import LongType, DoubleType, StringType, TimestampType

# ── Define the master schema we want ──────────────────────────────────────
def normalize_taxi_df(df):
    """Cast all columns to unified types and normalize column names."""
    
    # Normalize column names to lowercase
    df = df.toDF(*[c.lower() for c in df.columns])
    
    # Rename Airport_fee → airport_fee if it exists (already lowercased above)
    # Cast all int/bigint columns to LongType for consistency
    int_cols = ["vendorid", "pulocationid", "dolocationid", 
                "payment_type", "ratecodeid", "passenger_count"]
    
    for c in int_cols:
        if c in df.columns:
            df = df.withColumn(c, col(c).cast(LongType()))
    
    return df

# ── Read each file individually ───────────────────────────────────────────
jan = normalize_taxi_df(
    spark.read
         .option("spark.sql.parquet.enableVectorizedReader", "false")
         .parquet("/home/anantha/spark-etl-ml-project/data/raw/yellow_tripdata_2023-01.parquet")
)

feb = normalize_taxi_df(
    spark.read
         .option("spark.sql.parquet.enableVectorizedReader", "false")
         .parquet("/home/anantha/spark-etl-ml-project/data/raw/yellow_tripdata_2023-02.parquet")
)

mar = normalize_taxi_df(
    spark.read
         .option("spark.sql.parquet.enableVectorizedReader", "false")
         .parquet("/home/anantha/spark-etl-ml-project/data/raw/yellow_tripdata_2023-03.parquet")
)

# ── Check schemas match before union ──────────────────────────────────────
print("Jan schema:"); jan.printSchema()
print("Feb schema:"); feb.printSchema()
print("Mar schema:"); mar.printSchema()

Jan schema:
root
 |-- vendorid: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: long (nullable = true)
 |-- dolocationid: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

Feb schema:
root
 |-- vendorid: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpe

In [17]:
# Get common columns across all 3 files
jan_cols = set(jan.columns)
feb_cols = set(feb.columns)
mar_cols = set(mar.columns)

common_cols = sorted(jan_cols & feb_cols & mar_cols)

print(f"Common columns ({len(common_cols)}): {common_cols}")

taxi_df = jan.select(common_cols) \
             .union(feb.select(common_cols)) \
             .union(mar.select(common_cols))

print(f"\nTotal Taxi Records : {taxi_df.count():,}")
print(f"Columns used       : {len(common_cols)}")

Common columns (19): ['airport_fee', 'congestion_surcharge', 'dolocationid', 'extra', 'fare_amount', 'improvement_surcharge', 'mta_tax', 'passenger_count', 'payment_type', 'pulocationid', 'ratecodeid', 'store_and_fwd_flag', 'tip_amount', 'tolls_amount', 'total_amount', 'tpep_dropoff_datetime', 'tpep_pickup_datetime', 'trip_distance', 'vendorid']


[Stage 23:=================================================>      (21 + 3) / 24]


Total Taxi Records : 9,384,487
Columns used       : 19


In [18]:
print("=== TAXI DATA SCHEMA ===")
taxi_df.printSchema()

=== TAXI DATA SCHEMA ===
root
 |-- airport_fee: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- dolocationid: long (nullable = true)
 |-- extra: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- pulocationid: long (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- vendorid: long (nullable = true)



In [19]:
print("=== FIRST 5 TAXI ROWS ===")
taxi_df.show(5, truncate=False)

=== FIRST 5 TAXI ROWS ===
+-----------+--------------------+------------+-----+-----------+---------------------+-------+---------------+------------+------------+----------+------------------+----------+------------+------------+---------------------+--------------------+-------------+--------+
|airport_fee|congestion_surcharge|dolocationid|extra|fare_amount|improvement_surcharge|mta_tax|passenger_count|payment_type|pulocationid|ratecodeid|store_and_fwd_flag|tip_amount|tolls_amount|total_amount|tpep_dropoff_datetime|tpep_pickup_datetime|trip_distance|vendorid|
+-----------+--------------------+------------+-----+-----------+---------------------+-------+---------------+------------+------------+----------+------------------+----------+------------+------------+---------------------+--------------------+-------------+--------+
|0.0        |2.5                 |141         |1.0  |9.3        |1.0                  |0.5    |1              |2           |161         |1         |N            

In [20]:
print("=== TAXI DATA STATISTICS ===")
taxi_df.describe(
    "trip_distance", 
    "fare_amount", 
    "tip_amount", 
    "total_amount", 
    "passenger_count"
).show()

=== TAXI DATA STATISTICS ===


[Stage 28:=====================================================>  (23 + 1) / 24]

+-------+------------------+-----------------+------------------+------------------+------------------+
|summary|     trip_distance|      fare_amount|        tip_amount|      total_amount|   passenger_count|
+-------+------------------+-----------------+------------------+------------------+------------------+
|  count|           9384487|          9384487|           9384487|           9384487|           9148308|
|   mean| 3.874277528435664|18.51788001198273|3.4193540041140325| 27.26654346469178| 1.355499618071451|
| stddev|236.76264771847164|17.87964229785613|3.8930561193160576|22.326190988905786|0.8903758149364824|
|    min|               0.0|           -959.9|            -96.22|           -982.95|                 0|
|    max|         335004.33|           2203.1|             984.3|            2208.1|                 9|
+-------+------------------+-----------------+------------------+------------------+------------------+



In [21]:
from pyspark.sql.functions import col, sum as spark_sum, count

print("=== NULL COUNTS PER COLUMN ===")
null_counts = taxi_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in taxi_df.columns
])
null_counts.show(vertical=True)

=== NULL COUNTS PER COLUMN ===


[Stage 31:=====================================================>  (23 + 1) / 24]

-RECORD 0-----------------------
 airport_fee           | 236179 
 congestion_surcharge  | 236179 
 dolocationid          | 0      
 extra                 | 0      
 fare_amount           | 0      
 improvement_surcharge | 0      
 mta_tax               | 0      
 passenger_count       | 236179 
 payment_type          | 0      
 pulocationid          | 0      
 ratecodeid            | 236179 
 store_and_fwd_flag    | 236179 
 tip_amount            | 0      
 tolls_amount          | 0      
 total_amount          | 0      
 tpep_dropoff_datetime | 0      
 tpep_pickup_datetime  | 0      
 trip_distance         | 0      
 vendorid              | 0      



In [22]:
from pyspark.sql.functions import when

# Preview what our ML target will look like
# CLASSIFICATION: Did passenger tip more than 20%? (tip_pct > 0.20)
sample = taxi_df.withColumn(
    "tip_pct", 
    col("tip_amount") / col("fare_amount")
).withColumn(
    "generous_tipper",   # Our ML label
    when(col("tip_pct") > 0.20, 1).otherwise(0)
)

print("=== TIP DISTRIBUTION PREVIEW ===")
sample.groupBy("generous_tipper").count().show()

=== TIP DISTRIBUTION PREVIEW ===


+---------------+-------+
|generous_tipper|  count|
+---------------+-------+
|              1|5665644|
|              0|3718843|
+---------------+-------+



In [24]:
# Print a summary of what we know
print("=" * 50)
print("📊 DATASET SUMMARY")
print("=" * 50)
print(f"Total records  : {taxi_df.count():,}")
print(f"Total columns  : {len(taxi_df.columns)}")
print(f"Columns        : {taxi_df.columns}")
print(f"Date range     : Jan 2023 – Mar 2023")
# print(f"Zone lookup    : {zone_df.count()} NYC taxi zones")
print("=" * 50)
print("🎯 ML TASK: Classify generous tippers (tip > 20% of fare)")
print("=" * 50)

spark.stop()

📊 DATASET SUMMARY


Total records  : 9,384,487
Total columns  : 19
Columns        : ['airport_fee', 'congestion_surcharge', 'dolocationid', 'extra', 'fare_amount', 'improvement_surcharge', 'mta_tax', 'passenger_count', 'payment_type', 'pulocationid', 'ratecodeid', 'store_and_fwd_flag', 'tip_amount', 'tolls_amount', 'total_amount', 'tpep_dropoff_datetime', 'tpep_pickup_datetime', 'trip_distance', 'vendorid']
Date range     : Jan 2023 – Mar 2023
🎯 ML TASK: Classify generous tippers (tip > 20% of fare)
